# nb_semantic_model

Creates or updates the Direct Lake semantic model **NVIDIA Financials** on lakehouse **NVIDIALakehouse** (workspace **NVIDIA-BalanceSheet-AI**). Adds relationships, marks `Date` as the date table, and upserts the base measures on **Balance Sheet** (display folder `01-base`). No secrets. No calculated measure table.

**Kernel:** **Python** (not PySpark). Semantic Link / `sempy_labs` do not need Spark, and a Spark session can hit trial HTTP 430.

**Before Run all**

1. Attach default lakehouse **NVIDIALakehouse**.
2. Set the kernel to **Python**.
3. Capacity must allow **XMLA read/write** (needed for `connect_semantic_model(..., readonly=False)`). If that call fails, enable XMLA Read/Write on the trial capacity and rerun from the TOM cell — the generate step may already have created the model.
4. Gold tables must already exist, with date columns stored as **Date** (rerun `nb_bronze_silver_gold_sec_edgar` after the `to_date` write fix, then `nb_gold_dim_date` if you want its checks re-run): `gold.balance_sheet`, `gold.income_statement`, `gold.cash_flow`, `gold.dim_date`, `gold.dim_line_item`, `gold.dim_filing`.

Idempotent: if a model named exactly `NVIDIA Financials` already exists, this notebook **does not delete or overwrite it**. It updates relationships and measures in place via TOM.


In [ ]:
%pip install -q semantic-link-labs


In [ ]:
# Parameters
WORKSPACE = "NVIDIA-BalanceSheet-AI"
WORKSPACE_ID = "d1d09fde-24c7-4984-a4b0-eb55acf7c4f4"
LAKEHOUSE = "NVIDIALakehouse"
LAKEHOUSE_ID = "ece4bf8e-457a-4e4e-9628-4957de7a9c54"
DATASET = "NVIDIA Financials"
FY_CHECK = 2026
EXPECTED = {
    "Total Assets": 206_803_000_000,
    "Total Liabilities": 49_510_000_000,
    "Total Equity": 157_293_000_000,
    "Revenue": 215_938_000_000,
    "Net Income": 120_067_000_000,
}
# Friendly model table name → schema.table in the lakehouse
TABLES = {
    "Balance Sheet": "gold.balance_sheet",
    "Income Statement": "gold.income_statement",
    "Cash Flow": "gold.cash_flow",
    "Date": "gold.dim_date",
    "Line Item": "gold.dim_line_item",
    "Filing": "gold.dim_filing",
}
# Many → one, single-direction. Filing[reportDate] → Date is intentionally omitted (ambiguous path).
RELATIONSHIPS = [
    ("Balance Sheet", "period_end", "Date", "date"),
    ("Income Statement", "period_end", "Date", "date"),
    ("Cash Flow", "period_end", "Date", "date"),
    ("Balance Sheet", "concept", "Line Item", "concept"),
    ("Income Statement", "concept", "Line Item", "concept"),
    ("Cash Flow", "concept", "Line Item", "concept"),
    ("Balance Sheet", "accn", "Filing", "accn"),
    ("Income Statement", "accn", "Filing", "accn"),
    ("Cash Flow", "accn", "Filing", "accn"),
]


In [ ]:
from __future__ import annotations

import sempy.fabric as fabric
from sempy_labs.directlake import generate_direct_lake_semantic_model
from sempy_labs.tom import connect_semantic_model
from sempy_labs import refresh_semantic_model
from sempy_labs.lakehouse import lakehouse_attached

print("workspace:", WORKSPACE)
print("lakehouse attached:", lakehouse_attached())
if not lakehouse_attached():
    print(
        "WARN: attach NVIDIALakehouse as the default lakehouse, then rerun. "
        "The generate call still passes source/workspace explicitly."
    )


In [ ]:
# Inspect gold column types (expect Date for period_end / date after the gold write fix)
def _print_gold_types() -> None:
    try:
        from sempy_labs.lakehouse import get_lakehouse_columns

        cols = get_lakehouse_columns(lakehouse=LAKEHOUSE, workspace=WORKSPACE)
        print("get_lakehouse_columns rows:", len(cols))
        print(cols.head(20))
        names = {c.lower(): c for c in cols.columns}
        type_col = next((names[k] for k in ("type", "data type", "data_type") if k in names), None)
        table_col = next((names[k] for k in ("table name", "table", "name") if k in names), None)
        col_col = next((names[k] for k in ("column name", "column") if k in names), None)
        if type_col and table_col and col_col:
            mask = cols[col_col].isin(["period_end", "date", "concept", "accn"])
            print(cols.loc[mask, [table_col, col_col, type_col]].to_string(index=False))
    except Exception as ex:
        print("get_lakehouse_columns skipped:", type(ex).__name__, ex)


_print_gold_types()


## 1 · Create the model only if it is absent

`generate_direct_lake_semantic_model` is called with `overwrite=False` and `use_sql_endpoint=False` (Direct Lake on OneLake, not the SQL endpoint). If **NVIDIA Financials** already exists, this cell skips generation and the next cells update it via TOM.


In [ ]:
items = fabric.list_items(workspace=WORKSPACE)
# Column names vary slightly by sempy version
name_col = next(c for c in items.columns if c.lower() in {"display name", "name"})
type_col = next(c for c in items.columns if c.lower() == "type")
models = items[items[type_col].astype(str).str.lower().isin({"semanticmodel", "semantic model"})]
names = set(models[name_col].astype(str))
print("semantic models:", sorted(names) or "(none)")

created = False
if DATASET in names:
    print(f"Found existing '{DATASET}' — will update relationships/measures in place (no overwrite).")
else:
    extra = [n for n in names if n.lower() == DATASET.lower() and n != DATASET]
    if extra:
        raise SystemExit(
            f"A similarly named model exists {extra} but is not exactly '{DATASET}'. "
            "Refusing to create or overwrite. Rename it or set DATASET to match."
        )
    print(f"Creating Direct Lake model '{DATASET}' on {LAKEHOUSE} …")
    generate_direct_lake_semantic_model(
        dataset=DATASET,
        tables=TABLES,
        source=LAKEHOUSE,
        source_type="Lakehouse",
        source_workspace=WORKSPACE,
        use_sql_endpoint=False,
        workspace=WORKSPACE,
        refresh=True,
        inherit_descriptions=True,
        overwrite=False,
    )
    created = True
    print("generated + framed")

print("created_this_run:", created)


## 2 · TOM: relationships, date table, formats, measures

Requires XMLA read/write. Changes save when the `with connect_semantic_model(..., readonly=False)` block exits.


In [ ]:
MEASURES = [{'name': 'Total Assets', 'folder': '01-base', 'format': '#,##0', 'table': 'Balance Sheet', 'hidden': False, 'description': 'Point-in-time us-gaap Assets at the latest period_end in the current filter context (is_latest_filing only). Do not SUM across periods.', 'expr': 'VAR _end =\n    CALCULATE(\n        MAX(\'Balance Sheet\'[period_end]),\n        \'Balance Sheet\'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM(\'Balance Sheet\'[value_usd]),\n    \'Balance Sheet\'[concept] = "Assets",\n    \'Balance Sheet\'[is_latest_filing] = TRUE(),\n    \'Balance Sheet\'[period_end] = _end\n)'}, {'name': 'Total Liabilities', 'folder': '01-base', 'format': '#,##0', 'table': 'Balance Sheet', 'hidden': False, 'description': 'Point-in-time us-gaap Liabilities; latest period_end; is_latest_filing only.', 'expr': 'VAR _end =\n    CALCULATE(\n        MAX(\'Balance Sheet\'[period_end]),\n        \'Balance Sheet\'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM(\'Balance Sheet\'[value_usd]),\n    \'Balance Sheet\'[concept] = "Liabilities",\n    \'Balance Sheet\'[is_latest_filing] = TRUE(),\n    \'Balance Sheet\'[period_end] = _end\n)'}, {'name': 'Total Equity', 'folder': '01-base', 'format': '#,##0', 'table': 'Balance Sheet', 'hidden': False, 'description': 'Point-in-time us-gaap StockholdersEquity; latest period_end; is_latest_filing only.', 'expr': 'VAR _end =\n    CALCULATE(\n        MAX(\'Balance Sheet\'[period_end]),\n        \'Balance Sheet\'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM(\'Balance Sheet\'[value_usd]),\n    \'Balance Sheet\'[concept] = "StockholdersEquity",\n    \'Balance Sheet\'[is_latest_filing] = TRUE(),\n    \'Balance Sheet\'[period_end] = _end\n)'}, {'name': 'Cash', 'folder': '01-base', 'format': '#,##0', 'table': 'Balance Sheet', 'hidden': False, 'description': 'Point-in-time CashAndCashEquivalentsAtCarryingValue; latest period_end; is_latest_filing only.', 'expr': 'VAR _end =\n    CALCULATE(\n        MAX(\'Balance Sheet\'[period_end]),\n        \'Balance Sheet\'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM(\'Balance Sheet\'[value_usd]),\n    \'Balance Sheet\'[concept] = "CashAndCashEquivalentsAtCarryingValue",\n    \'Balance Sheet\'[is_latest_filing] = TRUE(),\n    \'Balance Sheet\'[period_end] = _end\n)'}, {'name': 'Long-Term Debt', 'folder': '01-base', 'format': '#,##0', 'table': 'Balance Sheet', 'hidden': False, 'description': 'Point-in-time LongTermDebtNoncurrent only (not total debt; current portion and leases are separate concepts and are not included).', 'expr': 'VAR _end =\n    CALCULATE(\n        MAX(\'Balance Sheet\'[period_end]),\n        \'Balance Sheet\'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM(\'Balance Sheet\'[value_usd]),\n    \'Balance Sheet\'[concept] = "LongTermDebtNoncurrent",\n    \'Balance Sheet\'[is_latest_filing] = TRUE(),\n    \'Balance Sheet\'[period_end] = _end\n)'}, {'name': 'Revenue', 'folder': '01-base', 'format': '#,##0', 'table': 'Balance Sheet', 'hidden': False, 'description': 'Duration revenue for the single latest period_end in context (is_latest_filing). Prefers RevenueFromContractWithCustomerExcludingAssessedTax when present, else Revenues. Not a sum of quarters. On a 10-Q this is the standalone fiscal quarter (gold prefers the 3-month row over YTD); on the FY 10-K it is the annual total.', 'expr': 'VAR _end =\n    CALCULATE(\n        MAX(\'Income Statement\'[period_end]),\n        \'Income Statement\'[is_latest_filing] = TRUE()\n    )\nVAR _contract =\n    CALCULATE(\n        SUM(\'Income Statement\'[value_usd]),\n        \'Income Statement\'[concept] = "RevenueFromContractWithCustomerExcludingAssessedTax",\n        \'Income Statement\'[is_latest_filing] = TRUE(),\n        \'Income Statement\'[period_end] = _end\n    )\nVAR _legacy =\n    CALCULATE(\n        SUM(\'Income Statement\'[value_usd]),\n        \'Income Statement\'[concept] = "Revenues",\n        \'Income Statement\'[is_latest_filing] = TRUE(),\n        \'Income Statement\'[period_end] = _end\n    )\nRETURN\nCOALESCE(_contract, _legacy)'}, {'name': 'Net Income', 'folder': '01-base', 'format': '#,##0', 'table': 'Balance Sheet', 'hidden': False, 'description': 'Duration NetIncomeLoss for the single latest period_end in context; is_latest_filing only. Not a sum of quarters. Standalone fiscal quarter on a 10-Q; annual total on the FY 10-K.', 'expr': 'VAR _end =\n    CALCULATE(\n        MAX(\'Income Statement\'[period_end]),\n        \'Income Statement\'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM(\'Income Statement\'[value_usd]),\n    \'Income Statement\'[concept] = "NetIncomeLoss",\n    \'Income Statement\'[is_latest_filing] = TRUE(),\n    \'Income Statement\'[period_end] = _end\n)'}, {'name': 'Operating Cash Flow', 'folder': '01-base', 'format': '#,##0', 'table': 'Balance Sheet', 'hidden': False, 'description': 'NetCashProvidedByUsedInOperatingActivities for the single latest period_end (is_latest_filing). 10-Q figures with duration_days > 100 are year-to-date (see [Cash Flow Is YTD]); they are kept raw — quarters are not derived.', 'expr': 'VAR _end =\n    CALCULATE(\n        MAX(\'Cash Flow\'[period_end]),\n        \'Cash Flow\'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM(\'Cash Flow\'[value_usd]),\n    \'Cash Flow\'[concept] = "NetCashProvidedByUsedInOperatingActivities",\n    \'Cash Flow\'[is_latest_filing] = TRUE(),\n    \'Cash Flow\'[period_end] = _end\n)'}, {'name': 'CapEx', 'folder': '01-base', 'format': '#,##0', 'table': 'Balance Sheet', 'hidden': True, 'description': 'Hidden helper: PaymentsToAcquirePropertyPlantAndEquipment at the latest period_end. NVIDIA companyfacts last carries this concept through mid-FY2020; later periods are blank.', 'expr': 'VAR _end =\n    CALCULATE(\n        MAX(\'Cash Flow\'[period_end]),\n        \'Cash Flow\'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM(\'Cash Flow\'[value_usd]),\n    \'Cash Flow\'[concept] = "PaymentsToAcquirePropertyPlantAndEquipment",\n    \'Cash Flow\'[is_latest_filing] = TRUE(),\n    \'Cash Flow\'[period_end] = _end\n)'}, {'name': 'Free Cash Flow', 'folder': '01-base', 'format': '#,##0', 'table': 'Balance Sheet', 'hidden': False, 'description': 'Operating Cash Flow minus CapEx (PaymentsToAcquirePropertyPlantAndEquipment). Returns BLANK when CapEx is missing (NVIDIA companyfacts drops this concept after mid-FY2020).', 'expr': 'VAR _ocf = [Operating Cash Flow]\nVAR _capex = [CapEx]\nRETURN\nIF(ISBLANK(_capex), BLANK(), _ocf - _capex)'}, {'name': 'Revenue YoY %', 'folder': '01-base', 'format': '0.0%', 'table': 'Balance Sheet', 'hidden': False, 'description': 'Revenue versus the same NVIDIA fiscal quarter in the prior fiscal year (matched on Date[fiscal_year]/[fiscal_quarter] quarter-end), not SAMEPERIODLASTYEAR.', 'expr': "VAR _curr = [Revenue]\nVAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 1,\n        'Date'[fiscal_quarter] = _fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Revenue],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nDIVIDE(_curr - _prior, _prior)"}, {'name': 'Net Income Margin %', 'folder': '01-base', 'format': '0.0%', 'table': 'Balance Sheet', 'hidden': False, 'description': '[Net Income] / [Revenue] for the same latest period_end in context.', 'expr': 'DIVIDE([Net Income], [Revenue])'}, {'name': 'Debt to Equity', 'folder': '01-base', 'format': '0.00', 'table': 'Balance Sheet', 'hidden': False, 'description': '[Long-Term Debt] (LongTermDebtNoncurrent only) / [Total Equity]. Not total debt.', 'expr': 'DIVIDE([Long-Term Debt], [Total Equity])'}, {'name': 'Cash Flow Is YTD', 'folder': '01-base', 'format': '0', 'table': 'Balance Sheet', 'hidden': False, 'description': '1 when the cash-flow row at the latest period_end in context is flagged is_ytd (10-Q with duration_days > 100). Operating Cash Flow and Free Cash Flow are raw YTD in that case.', 'expr': "VAR _end =\n    CALCULATE(\n        MAX('Cash Flow'[period_end]),\n        'Cash Flow'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    MAXX(VALUES('Cash Flow'[is_ytd]), IF('Cash Flow'[is_ytd], 1, 0)),\n    'Cash Flow'[is_latest_filing] = TRUE(),\n    'Cash Flow'[period_end] = _end\n)"}]

TABLE_ALIASES = {
    "Balance Sheet": ["Balance Sheet", "balance_sheet", "gold.balance_sheet", "gold_balance_sheet"],
    "Income Statement": ["Income Statement", "income_statement", "gold.income_statement", "gold_income_statement"],
    "Cash Flow": ["Cash Flow", "cash_flow", "gold.cash_flow", "gold_cash_flow"],
    "Date": ["Date", "dim_date", "gold.dim_date", "gold_dim_date"],
    "Line Item": ["Line Item", "dim_line_item", "gold.dim_line_item", "gold_dim_line_item"],
    "Filing": ["Filing", "dim_filing", "gold.dim_filing", "gold_dim_filing"],
}


def resolve_name(tom, logical: str) -> str:
    have = [t.Name for t in tom.model.Tables]
    for alias in TABLE_ALIASES[logical]:
        if alias in have:
            return alias
    raise KeyError(f"Table '{logical}' not in model. Have: {have}")


def col_dtype(tom, table: str, column: str) -> str:
    return str(tom.model.Tables[table].Columns[column].DataType)


def upsert_relationship(tom, from_table, from_col, to_table, to_col):
    for rel in list(tom.model.Relationships):
        if (
            rel.FromTable.Name == from_table
            and rel.FromColumn.Name == from_col
            and rel.ToTable.Name == to_table
            and rel.ToColumn.Name == to_col
        ):
            tom.remove_object(rel)
    tom.add_relationship(
        from_table=from_table,
        from_column=from_col,
        to_table=to_table,
        to_column=to_col,
        from_cardinality="Many",
        to_cardinality="One",
        cross_filtering_behavior="OneDirection",
        is_active=True,
    )


def upsert_measure(tom, host: str, m: dict) -> None:
    current = next((meas for meas in tom.all_measures() if meas.Name == m["name"]), None)
    if current is not None and current.Parent.Name != host:
        tom.remove_object(current)
        current = None
    if current is not None:
        tom.update_measure(
            measure_name=m["name"],
            expression=m["expr"],
            format_string=m["format"],
            hidden=m["hidden"],
            description=m["description"],
            display_folder=m["folder"],
        )
    else:
        tom.add_measure(
            table_name=host,
            measure_name=m["name"],
            expression=m["expr"],
            format_string=m["format"],
            hidden=m["hidden"],
            description=m["description"],
            display_folder=m["folder"],
        )


with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=False) as tom:
    # Ensure all Direct Lake tables exist (in case an older stub model is present)
    have = {t.Name for t in tom.model.Tables}
    missing = {}
    for logical, entity in TABLES.items():
        if not any(alias in have for alias in TABLE_ALIASES[logical]):
            missing[logical] = entity
    if missing:
        print("adding missing Direct Lake tables:", missing)
        tom.add_direct_lake_tables(
            tables=missing,
            source=LAKEHOUSE,
            source_type="Lakehouse",
            source_workspace=WORKSPACE,
            use_sql_endpoint=False,
        )

    resolved = {logical: resolve_name(tom, logical) for logical in TABLES}
    print("resolved tables:", resolved)

    # period_end must already be a date in Delta (same type as Date[date]).
    # Do not change model column types; they must match the lakehouse source.
    date_tbl = resolved["Date"]
    date_type = col_dtype(tom, date_tbl, "date")
    print(f"Date[date] type: {date_type}")
    for fact in ("Balance Sheet", "Income Statement", "Cash Flow"):
        ft = resolved[fact]
        pe_type = col_dtype(tom, ft, "period_end")
        print(f"{ft}[period_end] type: {pe_type}")
        if pe_type != date_type:
            raise SystemExit(
                f"{ft}[period_end] is {pe_type} but Date[date] is {date_type}. "
                "Rerun nb_bronze_silver_gold_sec_edgar so period_end is written as Date, "
                "then rerun this notebook."
            )

    for from_logical, from_col, to_logical, to_col in RELATIONSHIPS:
        upsert_relationship(tom, resolved[from_logical], from_col, resolved[to_logical], to_col)
        print(f"rel {from_logical}[{from_col}] → {to_logical}[{to_col}]")

    tom.mark_as_date_table(table_name=resolved["Date"], column_name="date", validate=False)

    # Sort-by, summarize, hide keys
    d = resolved["Date"]
    sort_pairs = [
        (d, "month_name", "month"),
        (d, "month_short", "month"),
        (d, "day_name", "day_of_week"),
        (d, "fiscal_quarter_label", "fiscal_year_quarter_sort"),
        (d, "fiscal_year_label", "fiscal_year"),
    ]
    for tbl, col, by in sort_pairs:
        if col in [c.Name for c in tom.model.Tables[tbl].Columns] and by in [c.Name for c in tom.model.Tables[tbl].Columns]:
            tom.set_sort_by_column(tbl, col, by)

    no_sum = {
        d: [
            "date_key", "year", "quarter", "month", "week_of_year", "day_of_month",
            "day_of_week", "fiscal_year", "fiscal_quarter", "fiscal_week",
            "day_of_fiscal_year", "fiscal_year_quarter_sort",
        ],
        resolved["Balance Sheet"]: ["fiscal_year", "duration_days"],
        resolved["Income Statement"]: ["fiscal_year", "duration_days"],
        resolved["Cash Flow"]: ["fiscal_year", "duration_days"],
    }
    for tbl, cols in no_sum.items():
        have_cols = {c.Name for c in tom.model.Tables[tbl].Columns}
        for c in cols:
            if c in have_cols:
                tom.set_summarize_by(tbl, c, "None")

    hide = {
        resolved["Balance Sheet"]: ["concept", "accn", "period_end", "value_usd", "ingest_date", "statement", "unit", "period_type"],
        resolved["Income Statement"]: ["concept", "accn", "period_end", "value_usd", "ingest_date", "statement", "unit", "period_type"],
        resolved["Cash Flow"]: ["concept", "accn", "period_end", "value_usd", "ingest_date", "statement", "unit", "period_type"],
        resolved["Line Item"]: ["concept"],
        resolved["Date"]: ["date_key", "fiscal_year_quarter_sort"],
    }
    for tbl, cols in hide.items():
        have_cols = {c.Name for c in tom.model.Tables[tbl].Columns}
        for c in cols:
            if c in have_cols:
                tom.model.Tables[tbl].Columns[c].IsHidden = True
                # Keep relationship / sort-by columns available in MDX
                if c in {"date_key", "ingest_date"}:
                    tom.set_is_available_in_mdx(tbl, c, False)

    # Keep Filing[accn] visible (citation key)
    for col in ("value_usd",):
        for fact in (resolved["Balance Sheet"], resolved["Income Statement"], resolved["Cash Flow"]):
            if col in [c.Name for c in tom.model.Tables[fact].Columns]:
                tom.model.Tables[fact].Columns[col].FormatString = "#,##0"
                tom.set_summarize_by(fact, col, "None")

    # Hierarchies (skip if already present)
    def ensure_hierarchy(table, name, columns):
        have_h = {h.Name for h in tom.model.Tables[table].Hierarchies}
        have_c = {c.Name for c in tom.model.Tables[table].Columns}
        if name in have_h:
            return
        if not all(c in have_c for c in columns):
            print("skip hierarchy", name, "missing columns")
            return
        tom.add_hierarchy(table_name=table, hierarchy_name=name, columns=columns)

    ensure_hierarchy(d, "Fiscal", ["fiscal_year_label", "fiscal_quarter_label", "date"])
    ensure_hierarchy(d, "Calendar", ["year", "month_name", "date"])

    # Host measures on a Direct Lake fact table (no calculated measure table).
    host = resolved["Balance Sheet"]

    # If an earlier draft created a calculated _Measures table, move measures off it and drop it.
    if any(t.Name == "_Measures" for t in tom.model.Tables):
        for m in list(tom.model.Tables["_Measures"].Measures):
            tom.remove_object(m)
        tom.remove_object(tom.model.Tables["_Measures"])
        print("removed calculated table _Measures")

    tom.model.DiscourageImplicitMeasures = True

    for m in MEASURES:
        upsert_measure(tom, host, m)
        print("measure:", m["name"], "on", host)

    print("TOM updates queued; saved on context exit")


## 3 · Frame (refresh) and validate FY2026


In [ ]:
refresh_semantic_model(dataset=DATASET, workspace=WORKSPACE, refresh_type="full")
print("refresh/framing complete")

# List relationships
with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=True) as tom:
    print("is_direct_lake:", tom.is_direct_lake())
    print("date table marked:", tom.is_date_table(next(t.Name for t in tom.all_date_tables()) if tom.has_date_table() else "?"))
    print("relationships:")
    for rel in tom.model.Relationships:
        print(
            f"  {rel.FromTable.Name}[{rel.FromColumn.Name}] → "
            f"{rel.ToTable.Name}[{rel.ToColumn.Name}] "
            f"active={rel.IsActive} xfilter={rel.CrossFilteringBehavior}"
        )
    print("measures:")
    for m in tom.all_measures():
        print(f"  {m.Parent.Name}[{m.Name}]")

dax = f"""
EVALUATE
ROW(
    "Total Assets", CALCULATE([Total Assets], 'Date'[fiscal_year] = {FY_CHECK}),
    "Total Liabilities", CALCULATE([Total Liabilities], 'Date'[fiscal_year] = {FY_CHECK}),
    "Total Equity", CALCULATE([Total Equity], 'Date'[fiscal_year] = {FY_CHECK}),
    "Revenue", CALCULATE([Revenue], 'Date'[fiscal_year] = {FY_CHECK}),
    "Net Income", CALCULATE([Net Income], 'Date'[fiscal_year] = {FY_CHECK})
)
"""
result = fabric.evaluate_dax(dataset=DATASET, workspace=WORKSPACE, dax_string=dax)
print(result)

# evaluate_dax column names are typically [Total Assets], sometimes with table prefix
row = result.iloc[0]
failed = []
for label, expected in EXPECTED.items():
    col = next((c for c in result.columns if label in str(c)), None)
    got = None if col is None else row[col]
    ok = col is not None and got is not None and abs(float(got) - float(expected)) <= 1
    status = "PASS" if ok else "FAIL"
    print(f"  {status} {label}: got={got} expected={expected:,.0f}")
    if not ok:
        failed.append(label)

print("OVERALL:", "FAIL" if failed else "PASS")
if failed:
    raise AssertionError("FY2026 check failed: " + ", ".join(failed))
